In [2]:
import numpy as np
import pandas as pd
import random
import torch
import matplotlib.pyplot as plt
%matplotlib inline


## Problem 1

In [3]:
df_path = "../dataset/homework/wine_train.csv"
df_org = pd.read_csv(df_path)


In [4]:
df = df_org.drop(columns='type')
# train_idx  = np.random.permutation(df.shape[0])
# train = df.iloc[train_idx[:int(0.8 * len(df))]]
# test  = df.iloc[train_idx[int(0.8 * len(df)):]]
# x_train = train.drop(columns=["quality"])
# y_train = train["quality"]
# x_test = test.drop(columns=["quality"])
# y_test = test["quality"]

In [5]:
def build_dataset(df, drop_col):
    x = df.drop(columns = drop_col)
    y = df[drop_col]
    return x, y

In [6]:
random.seed(42)
index = np.random.permutation(df.shape[0])
df = df.iloc[index]

In [7]:
n1 = int(0.8 * df.shape[0])
n2 = int(0.9 * df.shape[0])
x_train, y_train = build_dataset(df[:n1], "quality")
x_val, y_val = build_dataset(df[n1:n2], "quality" )
x_test, y_test = build_dataset(df[n2:], "quality" )

In [8]:
## sigmoid
def sigmoid(z):
    return 1/(1+np.exp(-z))

## relu
def my_relu(z):
    return np.maximum(0, z)

##  backprob
def backprob(x, y, x_test, y_test, 
             J = 100, learning_rate = 0.001, 
             epochs = 10, batch_size = 32, a = 0.01):
    x = np.asarray(x)
    y = np.asarray(y).reshape(-1, 1)
    x_test = np.asarray(x_test)
    y_test = np.asarray(y_test).reshape(-1, 1)
    x_test = np.column_stack([np.ones(x_test.shape[0]), x_test])
    ## add an 1 column for x for keep the bias
    x = np.column_stack([np.ones(x.shape[0]), x])
    y = y.reshape(-1, 1)
    n_x = x.shape[1]
    n_y = 1
    lambda_ = 0.8

    ## J is number of neurons
    ## initial the weight
    w = np.random.uniform(-a, a, (n_x, J)) ## n_x * J
    beta = np.random.uniform(-a, a, (J+1, n_y)) ## J+1 * n_y
    m_batch = batch_size
    m_total = x.shape[0]
    for _ in range(epochs):
        idx = np.random.permutation(m_total)
        x_shuffle = x[idx]
        y_shuffle = y[idx]
        for i in range(0, m_total, m_batch):
            x_m = x_shuffle[i: i + m_batch] ## minibatch x
            y_m = y_shuffle[i: i + m_batch] ## minibatch y
            m = x_m.shape[0]
            z = x_m @ w ## 1 hidden layer z is m * n_x * n_x * J = m * J
            h = sigmoid(z) ## m * J
            h_bias = np.column_stack([np.ones(h.shape[0]), h]) 
            y_pred = h_bias @ beta ## y_pred is m * J * J * 1

            ## gradient update
            beta_nonbias = beta[1:]  ## J * 1
            beta_penalty = np.copy(beta) 
            beta_penalty[0, :] = 0
            err = y_pred - y_m
            gbeta = (h_bias.T @ err) / m + 2 * lambda_ * beta_penalty ## J+1 * m * m * 1 averaged by m beta is J+1 * 1
            deltah    = err @ beta_nonbias.T * h * (1 - h) ## m * 1 * 1 * J + 1, m * J + 1
            gw = x_m.T @ deltah / m + 2 * lambda_ * w ## J * m * m * p
            beta = beta - learning_rate * gbeta
            w    = w - learning_rate * gw

        z_all = x @ w
        h_all = sigmoid(z_all)
        h_bias_all = np.column_stack([np.ones(h_all.shape[0]), h_all])
        y_est = h_bias_all @ beta

        ## compute the loss 
        beta_nonbias = beta[1:]
        loss = np.mean((y - y_est) ** 2) + lambda_ * (np.sum(w ** 2) + np.sum(beta_nonbias ** 2))
        print(f"Epoch {_+1}/{epochs}, Loss: {loss:.4f}")
    z_test = x_test @ w
    h_test = sigmoid(z_test)
    h_test_bias = np.column_stack([np.ones(h_test.shape[0]), h_test])
    y_test_pred = h_test_bias @ beta
    loss_test = np.mean((y_test - y_test_pred) ** 2)
    print(loss_test)
    return w, beta, loss, loss_test

In [9]:
# x_train_mean = x_train.mean(axis=0)
# x_train_std = x_train.std(axis=0)
# x_train_stdized = (x_train - x_train_mean) / x_train_std
# x_test_stdized = (x_test - x_train_mean) / x_train_std
w, beta, loss, loss_test = backprob(x_train.values, y_train.values, x_test.values, y_test.values, J=50, learning_rate=0.01, epochs=1000, batch_size=64, a=0.01)

Epoch 1/1000, Loss: 1.5159
Epoch 2/1000, Loss: 1.4681
Epoch 3/1000, Loss: 1.4375
Epoch 4/1000, Loss: 1.4122
Epoch 5/1000, Loss: 1.3851
Epoch 6/1000, Loss: 1.3551
Epoch 7/1000, Loss: 1.3329
Epoch 8/1000, Loss: 1.3120
Epoch 9/1000, Loss: 1.2911
Epoch 10/1000, Loss: 1.2853
Epoch 11/1000, Loss: 1.2479
Epoch 12/1000, Loss: 1.2291
Epoch 13/1000, Loss: 1.2066
Epoch 14/1000, Loss: 1.1921
Epoch 15/1000, Loss: 1.1784
Epoch 16/1000, Loss: 1.1871
Epoch 17/1000, Loss: 1.1405
Epoch 18/1000, Loss: 1.1657
Epoch 19/1000, Loss: 1.1096
Epoch 20/1000, Loss: 1.0925
Epoch 21/1000, Loss: 1.0803
Epoch 22/1000, Loss: 1.0626
Epoch 23/1000, Loss: 1.0526
Epoch 24/1000, Loss: 1.0332
Epoch 25/1000, Loss: 1.0204
Epoch 26/1000, Loss: 1.0149
Epoch 27/1000, Loss: 0.9986
Epoch 28/1000, Loss: 0.9856
Epoch 29/1000, Loss: 0.9746
Epoch 30/1000, Loss: 0.9650
Epoch 31/1000, Loss: 0.9539
Epoch 32/1000, Loss: 0.9526
Epoch 33/1000, Loss: 0.9737
Epoch 34/1000, Loss: 0.9349
Epoch 35/1000, Loss: 0.9238
Epoch 36/1000, Loss: 0.9217
E

In [10]:
from sklearn.linear_model import LinearRegression
model = LinearRegression().fit(x_train.values, y_train.values)
y_pred = model.predict(x_test.values)
mse_lr = np.mean((y_pred - y_test)**2)

In [11]:
## Problem 1.b
df = df_org.drop(columns='quality')
xby_train, yby_train = build_dataset(df[:n1], "type")
xby_val, yby_val = build_dataset(df[n1:n2], "type" )
xby_test, yby_test = build_dataset(df[n2:], "type" )


In [12]:
## backprob of type
def backprob_two_class(x, y, 
                       x_val, y_val,
                       lr = 0.001, batch_size = 32, 
                       epochs = 100,
                       J = 100, a = 0.001):
    x     = np.asarray(x)
    y     = np.asarray(y)
    x_val = np.asarray(x_val)
    y_val = np.asarray(y_val)

    np.random.seed(42)
    n_x   = x.shape[1]
    n     = x.shape[0]
    ## need to change y to one hot
    y     = np.where(y == "white", 1, 0)
    y     = y.reshape(-1, 1)
    y_val = np.where(y_val == "white", 1, 0)
    y_val = y_val.reshape(-1, 1)
    n_y   = y.shape[1]
    m = batch_size
    w     = np.random.uniform(-a, a, size=[n_x, J])
    bw    = np.random.uniform(-a, a, size=J)
    beta  = np.random.uniform(-a, a, size = [J, n_y])
    bb    = np.random.uniform(-a, a, size = 1)
    gbeta = []; gw = []; gbb = 0; gbw = 0
    for epoch in range(epochs):
        idx       = np.random.permutation(n)
        x_shuffle = x[idx]
        y_shuffle = y[idx]
        for i in range(0, n, batch_size):
            x_m    = x_shuffle[i:m+i,]
            y_m    = y_shuffle[i:m+i,]
            ## value for first layer
            h1     = x_m @ w + bw 
            z      = sigmoid(h1)
            h2     = z @ beta + bb
            y_pred = sigmoid(h2)
            ## gradient of weight and bias
            err    = y_pred - y_m
            gbeta  = z.T @ err / x_m.shape[0]
            gbb    = np.sum(err)  / x_m.shape[0] ## just a number
            deltaz = err @ beta.T * (z * (1 - z)) 
            gw     = x_m.T @ deltaz / x_m.shape[0]
            gbw    = np.sum(deltaz, axis=0) / x_m.shape[0]

            ## update
            beta -= lr * gbeta
            w    -= lr * gw
            bb   -= lr * gbb
            bw   -= lr * gbw

        ## compute the loss for all data
        eps = 1e-12
        z_all      = sigmoid(x @ w + bw)
        h2_all     = z_all @ beta + bb
        y_pred_all = sigmoid(h2_all)
        # loss       = - np.mean(y.T @ np.log(y_pred_all + eps) + 
        #                  (1 - y).T @ np.log(1 - y_pred_all + eps))
        loss       = -np.mean(y * np.log(y_pred_all + eps) +
                        (1 - y) * np.log(1 - y_pred_all + eps))
        # print(f"Epoch {epoch+1}/{epochs}, Loss: {loss:.6f}")
    
    z_test      = sigmoid(x_val @ w + bw)
    h2_test     = z_test @ beta + bb
    y_pred_test = sigmoid(h2_test)
    # loss_test   = - np.mean(y_val.T @ np.log(y_pred_test + eps) + 
    #                       (1 - y_val).T @ np.log(1 - y_pred_test + eps))
    loss_test   = -np.mean(y_val * np.log(y_pred_test + eps) +
                        (1 - y_val) * np.log(1 - y_pred_test + eps))
    y_hat_test  = (y_pred_test >= 0.5).astype(int)
    acc_test    = np.mean(y_hat_test == y_val)
    return w, beta, bb, bw, loss_test, acc_test

In [13]:
w_, beta_, bb_, bw_, loss_, acc_test_ = backprob_two_class(xby_train.values, yby_train.values, xby_val.values, yby_val.values, lr=0.01, batch_size=32, epochs=1000, J=100, a=0.01)

In [14]:
print(f"Final Loss on Validation Set: {loss_:.6f}",
    f"Final Accuracy on Validation Set: {acc_test_:.6f}")

Final Loss on Validation Set: 0.074082 Final Accuracy on Validation Set: 0.980658


## Problem 2
First we apply the Tensorflow version


### Example from slides

In [15]:
## Build one neural network mode with two output layer
## import modules
import tensorflow as tf
import numpy as np

In [16]:
## example from slides
num_samples = 10_000
x_train     = np.random.uniform(0, 4.5, size=(num_samples, 1))
y_train     = x_train**2 + np.random.normal(size=(num_samples, 1)) 
## this is predict the y = x^2
inputs  = tf.keras.Input(shape=(1,))## this is the (dimension of inputs, size of inputs)
hidden1 = tf.keras.layers.Dense(32, activation='relu')(inputs)
hidden2 = tf.keras.layers.Dense(16, activation='relu')(hidden1)
outputs = tf.keras.layers.Dense(1)(hidden2)

model   = tf.keras.Model(inputs = inputs, outputs = outputs)

In [17]:
model.compile(optimizer="adam", loss="mse")
model.fit(x_train, y_train, epochs=10)

x_test = np.array(range(1, 5))
y_pred = model.predict(x_test)

print(x_test**2 - y_pred[:, 0])

Epoch 1/10
313/313 ━━━━━━━━━━━━━━━━━━━━ 1s 309us/step - loss: 35.8685 
Epoch 2/10
313/313 ━━━━━━━━━━━━━━━━━━━━ 0s 290us/step - loss: 4.4131
Epoch 3/10
313/313 ━━━━━━━━━━━━━━━━━━━━ 0s 312us/step - loss: 2.0196
Epoch 4/10
313/313 ━━━━━━━━━━━━━━━━━━━━ 0s 292us/step - loss: 1.3427
Epoch 5/10
313/313 ━━━━━━━━━━━━━━━━━━━━ 0s 300us/step - loss: 1.1700
Epoch 6/10
313/313 ━━━━━━━━━━━━━━━━━━━━ 0s 395us/step - loss: 1.1120
Epoch 7/10
313/313 ━━━━━━━━━━━━━━━━━━━━ 0s 292us/step - loss: 1.0842
Epoch 8/10
313/313 ━━━━━━━━━━━━━━━━━━━━ 0s 293us/step - loss: 1.0717
Epoch 9/10
313/313 ━━━━━━━━━━━━━━━━━━━━ 0s 289us/step - loss: 1.0633
Epoch 10/10
313/313 ━━━━━━━━━━━━━━━━━━━━ 0s 290us/step - loss: 1.0601
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 24ms/step
[0.04057866 0.21807265 0.29815674 0.3968401 ]


In [18]:
## Dual output model
## example from slides
num_samples = 10_000
x_train     = np.random.uniform(0, 4.5, size=(num_samples, 1))
y1_train    = x_train**2 + np.random.normal(size=(num_samples, 1)) 
y2_train    = 0.5*x_train + np.random.normal(size=(num_samples, 1))
Y = np.hstack([y1_train, y2_train])

## define the nn
inputs    = tf.keras.Input(shape=(1,))
hidden1   = tf.keras.layers.Dense(32, activation="relu")(inputs)
hidden2   = tf.keras.layers.Dense(16, activation="relu")(hidden1)
outputs   = tf.keras.layers.Dense(2, name="output")(hidden2)
dualmodel = tf.keras.Model(inputs = inputs, outputs = outputs)

In [19]:
losses = {"output": "mse"}
dualmodel.compile(optimizer='adam', loss=losses) ## this will give the average loss based on the loss_weights

dualmodel.fit(x_train, Y, epochs=10)

x_test = np.array(range(1, 6))
dualmodel.predict(x_test)

Epoch 1/10
313/313 ━━━━━━━━━━━━━━━━━━━━ 1s 339us/step - loss: 9.5691 
Epoch 2/10
313/313 ━━━━━━━━━━━━━━━━━━━━ 0s 309us/step - loss: 2.4286
Epoch 3/10
313/313 ━━━━━━━━━━━━━━━━━━━━ 0s 324us/step - loss: 1.4744
Epoch 4/10
313/313 ━━━━━━━━━━━━━━━━━━━━ 0s 319us/step - loss: 1.2320
Epoch 5/10
313/313 ━━━━━━━━━━━━━━━━━━━━ 0s 317us/step - loss: 1.1133
Epoch 6/10
313/313 ━━━━━━━━━━━━━━━━━━━━ 0s 325us/step - loss: 1.0462
Epoch 7/10
313/313 ━━━━━━━━━━━━━━━━━━━━ 0s 322us/step - loss: 1.0222
Epoch 8/10
313/313 ━━━━━━━━━━━━━━━━━━━━ 0s 316us/step - loss: 1.0140
Epoch 9/10
313/313 ━━━━━━━━━━━━━━━━━━━━ 0s 287us/step - loss: 1.0103
Epoch 10/10
313/313 ━━━━━━━━━━━━━━━━━━━━ 0s 285us/step - loss: 1.0099
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 23ms/step


array([[ 1.0764713,  0.520538 ],
       [ 3.8983073,  0.9917818],
       [ 8.839786 ,  1.4187375],
       [15.861237 ,  1.91828  ],
       [22.986326 ,  2.4116268]], dtype=float32)

### Problem 2 Using Tensorflow

In [20]:
## Build one neural network mode with two output layer
## import modules
import tensorflow as tf
import numpy as np

In [21]:
wine_file      = "../dataset/homework/wine_train.csv"
wine_test_file = "../dataset/homework/wine_test.csv"
wine_train     = pd.read_csv(wine_file)
wine_test      = pd.read_csv(wine_test_file)

In [24]:
x_train    = wine_train.drop(columns=["quality", "type"])
y1_train   = wine_train["quality"] ## quality is linear
y2_train   = np.where(wine_train["type"] == "white", 1, 0) ## type is quadratic
Y_dict = {
    "reg_out" : y1_train,
    "type_out": y2_train
}


In [25]:
inputs = tf.keras.Input(shape=(11,))
x = tf.keras.layers.Dense(64, activation="relu")(inputs)
# x = tf.keras.layers.Dropout(0.2)(x)
x = tf.keras.layers.BatchNormalization()(x)
x = tf.keras.layers.Dense(32, activation="relu")(x)
# x = tf.keras.layers.Dropout(0.2)(x)
x = tf.keras.layers.BatchNormalization()(x)
x = tf.keras.layers.Dense(64, activation="relu")(x)
reg_output  = tf.keras.layers.Dense(1, name="reg_out")(x)
type_output = tf.keras.layers.Dense(1, activation="sigmoid", name="type_out")(x)
model = tf.keras.Model(inputs = inputs, outputs = [reg_output, type_output])
losses = {"reg_out" : "mse",
          "type_out": "binary_crossentropy"}
loss_weights = {"reg_out": 0.5, "type_out": 0.5}
metrics_dict = {
    "reg_out": "mae",       
    "type_out": "accuracy"
}
def lr_schedule(epoch):
    lr = 0.01
    if epoch > 5:
        lr = 0.001
    return lr
lr_callback = tf.keras.callbacks.LearningRateScheduler(lr_schedule)
model.compile(optimizer="adam",
              loss=losses, loss_weights=loss_weights,
              metrics=metrics_dict)

early_stopping_callback = tf.keras.callbacks.EarlyStopping(monitor="val_loss", patience=10)

In [26]:
model.fit(
    x_train, 
    Y_dict,
    epochs=1000,
    batch_size=32,
    validation_split=0.2,
    callbacks=[early_stopping_callback, lr_callback]
)

Epoch 1/1000
130/130 ━━━━━━━━━━━━━━━━━━━━ 2s 3ms/step - loss: 1.1700 - reg_out_loss: 2.0555 - reg_out_mae: 0.9897 - type_out_accuracy: 0.8854 - type_out_loss: 0.2751 - val_loss: 1.5317 - val_reg_out_loss: 2.9322 - val_reg_out_mae: 1.4029 - val_type_out_accuracy: 0.9565 - val_type_out_loss: 0.1528 - learning_rate: 0.0100
Epoch 2/1000
130/130 ━━━━━━━━━━━━━━━━━━━━ 0s 751us/step - loss: 0.4825 - reg_out_loss: 0.7848 - reg_out_mae: 0.6917 - type_out_accuracy: 0.9354 - type_out_loss: 0.1806 - val_loss: 1.5751 - val_reg_out_loss: 2.6551 - val_reg_out_mae: 1.3592 - val_type_out_accuracy: 0.7408 - val_type_out_loss: 0.5269 - learning_rate: 0.0100
Epoch 3/1000
130/130 ━━━━━━━━━━━━━━━━━━━━ 0s 736us/step - loss: 0.4240 - reg_out_loss: 0.7045 - reg_out_mae: 0.6555 - type_out_accuracy: 0.9449 - type_out_loss: 0.1450 - val_loss: 1.6269 - val_reg_out_loss: 1.9517 - val_reg_out_mae: 1.1314 - val_type_out_accuracy: 0.5329 - val_type_out_loss: 1.3198 - learning_rate: 0.0100
Epoch 4/1000
130/130 ━━━━━━━━━

In [ ]:
model.predict(wine_test)


21/21 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step 


[array([[6.012535 ],
        [5.837843 ],
        [6.3296585],
        [5.6348705],
        [4.7880616],
        [5.197678 ],
        [6.7309227],
        [6.488416 ],
        [6.3726964],
        [5.639028 ],
        [6.030237 ],
        [6.15898  ],
        [6.509468 ],
        [5.7982764],
        [5.423675 ],
        [5.9889574],
        [5.870061 ],
        [5.873376 ],
        [6.2812066],
        [5.700536 ],
        [6.2660923],
        [6.3598013],
        [5.4078035],
        [4.7354054],
        [5.5006037],
        [6.1610875],
        [5.6122465],
        [6.646581 ],
        [5.999858 ],
        [4.8198824],
        [5.1662903],
        [5.958358 ],
        [6.91387  ],
        [5.5503073],
        [6.800956 ],
        [5.6997886],
        [6.468482 ],
        [6.424089 ],
        [6.0253096],
        [5.638341 ],
        [6.5740366],
        [7.4106007],
        [5.6698956],
        [5.079546 ],
        [5.802991 ],
        [6.262911 ],
        [6.3904257],
        [5.17

### Using PyTorch

In [27]:
import torch
import torch.nn.functional as F
import torch.optim as optim
from torch import nn
from torch.utils.data import DataLoader
from torch.utils.data import TensorDataset

In [28]:
device = torch.accelerator.current_accelerator().type if torch.accelerator.is_available() else "cpu"

In [29]:
class MultiTaskModel(nn.Module):
    def __init__(self, input_dim, hidden_dim):
        super().__init__()
        # self.flatten = nn.Flatten()
        self.shared_features = nn.Sequential(
            nn.Linear(input_dim, hidden_dim),
            nn.BatchNorm1d(hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, hidden_dim),
            nn.BatchNorm1d(hidden_dim),
            nn.ReLU(),
        )
        self.reg_head  = nn.Linear(hidden_dim, 1)
        self.type_head = nn.Linear(hidden_dim, 1)
    
    def forward(self, x):
        features = self.shared_features(x)
        reg_out  = self.reg_head(features)
        type_out = self.type_head(features)

        return reg_out, type_out

In [30]:
model_torch = MultiTaskModel(11, 64)


In [31]:
x_tensor      = torch.tensor(x_train.values,  dtype=torch.float32)
y_reg_tensor  = torch.tensor(y1_train.values, dtype=torch.float32).view(-1, 1)
y_type_tensor = torch.tensor(y2_train, dtype=torch.float32).view(-1, 1)

dataset       = TensorDataset(x_tensor, y_reg_tensor, y_type_tensor)

In [32]:
batch_size = 32
train_loader = DataLoader(dataset=dataset, batch_size=batch_size, shuffle = True)

In [38]:
batch_size = 32
train_loader   = DataLoader(dataset=dataset, batch_size=batch_size, shuffle = True)
optimizer      = optim.Adam(model_torch.parameters(), lr = 0.001)
criterion_reg  = nn.MSELoss()
criterion_type = nn.BCEWithLogitsLoss()
for epoch in range(100):
    model_torch.train()
    epoch_total_loss = 0.0
    epoch_reg_loss = 0.0
    epoch_type_loss = 0.0
    for batch_x, batch_y_reg, batch_y_type in train_loader:
        optimizer.zero_grad()
        pred_reg, pred_type = model_torch(batch_x)

        loss_reg = criterion_reg(pred_reg, batch_y_reg)
        loss_type = criterion_type(pred_type, batch_y_type)

        loss = 0.5 * loss_reg + 0.5 * loss_type
        loss.backward()
        optimizer.step()

        epoch_total_loss += loss.item() * batch_x.size(0)
        epoch_reg_loss += loss_reg.item() * batch_x.size(0)
        epoch_type_loss += loss_type.item() * batch_x.size(0)

    n = len(train_loader.dataset)

    print(
        f"epoch {epoch + 1:03d} | "
        f"total={epoch_total_loss / n:.4f} | "
        f"reg={epoch_reg_loss / n:.4f} | "
        f"type={epoch_type_loss / n:.4f}"
    )

epoch 001 | total=0.3381 | reg=0.6038 | type=0.0723
epoch 002 | total=0.3348 | reg=0.6059 | type=0.0637
epoch 003 | total=0.3197 | reg=0.5753 | type=0.0642
epoch 004 | total=0.3161 | reg=0.5724 | type=0.0597
epoch 005 | total=0.3168 | reg=0.5663 | type=0.0674
epoch 006 | total=0.3206 | reg=0.5790 | type=0.0622
epoch 007 | total=0.3050 | reg=0.5547 | type=0.0553
epoch 008 | total=0.3074 | reg=0.5561 | type=0.0588
epoch 009 | total=0.3143 | reg=0.5680 | type=0.0606
epoch 010 | total=0.3105 | reg=0.5582 | type=0.0628
epoch 011 | total=0.3010 | reg=0.5452 | type=0.0568
epoch 012 | total=0.3019 | reg=0.5479 | type=0.0560
epoch 013 | total=0.2994 | reg=0.5465 | type=0.0522
epoch 014 | total=0.3097 | reg=0.5619 | type=0.0575
epoch 015 | total=0.3008 | reg=0.5432 | type=0.0583
epoch 016 | total=0.3018 | reg=0.5525 | type=0.0512
epoch 017 | total=0.2973 | reg=0.5440 | type=0.0506
epoch 018 | total=0.2901 | reg=0.5273 | type=0.0529
epoch 019 | total=0.2931 | reg=0.5357 | type=0.0506
epoch 020 | 

In [51]:
x_test_tensor = torch.tensor(wine_test.values, dtype=torch.float32)
model_torch.eval()

with torch.no_grad():
    pred_quality, pred_type_logit = model_torch(x_test_tensor)
    pred_type_prob = torch.sigmoid(pred_type_logit)
    pred_type_label = (pred_type_prob >= 0.5).long()

In [52]:
def prediction_model(model, x_tensor, batch_size=256):
    model.eval()

    all_reg_preds = []
    all_type_preds = []

    test_dataset = TensorDataset(x_tensor)
    test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False)

    with torch.no_grad():
        for batch in test_loader:
            x_batch = batch[0]

            reg_out, type_logit = model(x_batch)

            type_prob = torch.sigmoid(type_logit)
            type_pred = (pred_type_prob >= 0.5).long()

            all_reg_preds.append(reg_out.numpy())
            all_type_preds.append(type_pred.numpy())

    final_reg_preds = np.vstack(all_reg_preds)
    final_type_preds = np.vstack(all_type_preds)

    return final_reg_preds, final_type_preds    

In [ ]:
pred_quality_np, pred_type_prob_np = prediction_model(model_torch, x_test_tensor, batch_size=256)

pred_type_label_np = (pred_type_prob_np >= 0.5).astype(int)

In [54]:
print(pred_type_label_np)

[[1]
 [1]
 [1]
 ...
 [1]
 [1]
 [1]]
